In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Chuẩn bị dữ liệu
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. Xây dựng model CNN cho CIFAR-10
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Ảnh đầu vào 3 kênh màu (RGB), kích thước 32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 32x32 -> 32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 16x16 -> 16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # 8x8 -> 8x8

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # Ảnh 32x32 qua 3 lần pool (chia 2) -> kích thước còn 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model_cifar = CIFAR10_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cifar.parameters(), lr=0.001)

# Số lần lặp qua toàn bộ tập dữ liệu (Epochs)
num_epochs = 10

print("Bắt đầu quá trình huấn luyện CIFAR-10...")

for epoch in range(num_epochs):
    model_cifar.train() # Đặt mô hình ở chế độ huấn luyện (kích hoạt Dropout)
    running_loss = 0.0
    correct = 0
    total = 0

    for i, data in enumerate(train_loader, 0):
        # Lấy dữ liệu (ảnh và nhãn) và chuyển vào device (GPU/CPU)
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Reset gradient về 0 sau mỗi bước
        optimizer.zero_grad()

        # 2. Forward pass: Đưa ảnh qua mô hình để lấy dự đoán
        outputs = model_cifar(inputs)

        # 3. Tính toán độ lỗi (Loss)
        loss = criterion(outputs, labels)

        # 4. Backward pass: Lan truyền ngược để tính đạo hàm
        loss.backward()

        # 5. Cập nhật trọng số của mô hình
        optimizer.step()

        # --- Phần này chỉ để tính toán thống kê in ra màn hình ---
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Tính loss và độ chính xác trung bình cho cả Epoch
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = 100 * correct / total

    print(f'Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

print('Hoàn thành huấn luyện!')


100%|██████████| 170M/170M [52:31<00:00, 54.1kB/s]


Bắt đầu quá trình huấn luyện CIFAR-10...
Epoch [1/10] | Train Loss: 1.5847 | Train Acc: 41.86%
Epoch [2/10] | Train Loss: 1.1908 | Train Acc: 57.20%
Epoch [3/10] | Train Loss: 1.0117 | Train Acc: 64.31%
Epoch [4/10] | Train Loss: 0.9121 | Train Acc: 67.86%
Epoch [5/10] | Train Loss: 0.8403 | Train Acc: 70.48%
Epoch [6/10] | Train Loss: 0.7990 | Train Acc: 71.94%
Epoch [7/10] | Train Loss: 0.7538 | Train Acc: 73.70%
Epoch [8/10] | Train Loss: 0.7220 | Train Acc: 74.95%
Epoch [9/10] | Train Loss: 0.6931 | Train Acc: 75.93%
Epoch [10/10] | Train Loss: 0.6731 | Train Acc: 76.57%
Hoàn thành huấn luyện!
